## Pod IPs & pod CIDRs — where the addresses come from

A kubeadm cluster has two ranges, separate from your VPC/LAN:

- **Pod CIDR** — where pods get IPs. `192.168.0.0/16` (kubeadm/Calico), `10.244.0.0/16` (Flannel).
- **Service CIDR** — where Service ClusterIPs come from. `10.96.0.0/12` (notebook 04).

The pod CIDR is **subdivided per node** — each node gets a slice (typically a `/24`, 256 IPs) to allocate from:

```
cluster pod CIDR 192.168.0.0/16
  ├── node-1 → 192.168.1.0/24 → pods 192.168.1.5, .6, .7 …
  └── node-2 → 192.168.2.0/24 → pods 192.168.2.4, .5 …
```

`kubectl get node <name> -o jsonpath='{.spec.podCIDR}'` shows a node's slice. This layout lets the cluster **route by node**: "any packet for `192.168.2.0/24` → node-2." The CNI's job is to publish those per-node routes — via BGP, overlay tunnel headers, or whatever it chose.

**The everyday gotcha:** `192.168.0.0/16` overlaps with corporate LANs and home routers. If your `kind` cluster picks the same range as your VPN, `192.168.x.y` from the host is ambiguous — cluster or office? Production clusters pick non-overlapping ranges. On our map, pod IPs live on the **Pods** box; this section is why each Pod there has a real, node-routable address rather than a shared host port.